# QGT 计算诊断和修复

问题：QGT trace = 0，说明梯度计算有问题

可能原因：
1. 梯度展平方式不正确
2. 复数梯度处理有问题
3. 中心化计算错误

In [ ]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
from functools import partial

# 设置系统
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
E_fci = cisolver.kernel()[0]

ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(n_orbitals=2, s=1/2, n_fermions_per_spin=(1,1))

class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

def forward(GraphDef_State, x):
    log_psi, _ = nnx.call(GraphDef_State)(x)
    return log_psi

# 初始化模型
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(n_spin_orbitals=4, hidden_dim=12, rngs=rngs)
graphdef, model_state = nnx.split(model)

print("模型初始化完成")

## 诊断 1：检查梯度计算

In [ ]:
# 生成一些测试样本
edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=16, sweep_size=32)

GraphDef_State = (graphdef, model_state)
sampler_state = sampler.init_state(forward, GraphDef_State, seed=21)
sampler_state = sampler.reset(forward, GraphDef_State, state=sampler_state)
samples, sampler_state = sampler.sample(forward, GraphDef_State, state=sampler_state, chain_length=64)
samples = samples.reshape(-1, 4)

print(f"样本形状: {samples.shape}")
print(f"样本前 5 个: \n{samples[:5]}")

In [ ]:
# 计算单个样本的梯度
def logpsi_single_param(s, x):
    return forward((graphdef, s), x)

# 方法 1：使用 holomorphic=True
grad1 = jax.grad(logpsi_single_param, argnums=0, holomorphic=True)(model_state, samples[0])

# 方法 2：不使用 holomorphic（需要处理实部和虚部）
# grad2 = jax.grad(lambda s, x: logpsi_single_param(s, x).real, argnums=0)(model_state, samples[0])

print("梯度结构:")
for key, value in grad1.items():
    if isinstance(value, dict):
        for subkey, subvalue in value.items():
            print(f"  {key}.{subkey}: shape={subvalue.shape}, dtype={subvalue.dtype}")
            print(f"    范数: {jnp.linalg.norm(subvalue):.6e}")
            print(f"    实部范围: [{jnp.min(subvalue.real):.6e}, {jnp.max(subvalue.real):.6e}]")
            print(f"    虚部范围: [{jnp.min(subvalue.imag):.6e}, {jnp.max(subvalue.imag):.6e}]")

## 诊断 2：检查梯度展平

In [ ]:
# 批量计算梯度
def grad_logpsi_single(x):
    return jax.grad(logpsi_single_param, argnums=0, holomorphic=True)(model_state, x)

grads_tree = jax.vmap(grad_logpsi_single)(samples)

print("梯度树结构:")
print(f"  类型: {type(grads_tree)}")
print(f"  键: {grads_tree.keys()}")

# 展平梯度
grads_flat, tree_def = jax.tree_util.tree_flatten(grads_tree)

print(f"\n展平后的梯度:")
print(f"  数量: {len(grads_flat)}")
for i, g in enumerate(grads_flat):
    print(f"  梯度 {i}: shape={g.shape}, dtype={g.dtype}")
    print(f"    范数: {jnp.linalg.norm(g):.6e}")

# 连接梯度
n_samples = samples.shape[0]
grads_concat = jnp.concatenate([g.reshape((n_samples, -1)) for g in grads_flat], axis=-1)

print(f"\n连接后的梯度矩阵:")
print(f"  形状: {grads_concat.shape}")
print(f"  范数: {jnp.linalg.norm(grads_concat):.6e}")
print(f"  每行的范数（前 5 个）:")
for i in range(min(5, n_samples)):
    print(f"    样本 {i}: {jnp.linalg.norm(grads_concat[i]):.6e}")

## 诊断 3：检查 QGT 计算

In [ ]:
# 计算均值和中心化
mean_grad = jnp.mean(grads_concat, axis=0, keepdims=True)
centered_grads = grads_concat - mean_grad

print("中心化前:")
print(f"  梯度矩阵范数: {jnp.linalg.norm(grads_concat):.6e}")
print(f"  均值向量范数: {jnp.linalg.norm(mean_grad):.6e}")

print("\n中心化后:")
print(f"  中心化梯度范数: {jnp.linalg.norm(centered_grads):.6e}")
print(f"  中心化梯度均值: {jnp.mean(jnp.abs(centered_grads)):.6e}")

# 检查是否所有梯度都相同
grad_diff = jnp.max(jnp.abs(centered_grads))
print(f"\n中心化梯度的最大绝对值: {grad_diff:.6e}")

if grad_diff < 1e-10:
    print("⚠️ 警告：所有样本的梯度几乎相同！")
    print("   这说明波函数在参数空间上几乎是平坦的。")

In [ ]:
# 计算 QGT
qgt = (1.0 / n_samples) * jnp.einsum('si,sj->ij', centered_grads.conj(), centered_grads)

print("QGT 诊断:")
print(f"  形状: {qgt.shape}")
print(f"  trace: {jnp.trace(qgt):.6e}")
print(f"  行列式: {jnp.linalg.det(qgt):.6e}")
print(f"  条件数: {jnp.linalg.cond(qgt):.6e}")
print(f"  秩: {jnp.linalg.matrix_rank(qgt)}")
print(f"  对角线元素范围: [{jnp.min(jnp.abs(jnp.diag(qgt))):.6e}, {jnp.max(jnp.abs(jnp.diag(qgt))):.6e}]")

# 检查 QGT 是否为零矩阵
qgt_norm = jnp.linalg.norm(qgt)
print(f"\nQGT 范数: {qgt_norm:.6e}")

if qgt_norm < 1e-10:
    print("⚠️ 严重警告：QGT 几乎为零矩阵！")
    print("   这意味着参数空间中几乎没有变化。")

## 诊断 4：与 NetKet 对比

In [ ]:
# 使用 NetKet 计算 QGT
vstate = nk.vqs.MCState(
    sampler=sampler,
    model=model,
    n_samples=1000,
    n_discard_per_chain=10
)

# NetKet 的 QGT
qgt_netket = vstate.quantum_geometric_tensor(nk.optimizer.SR(diag_shift=0.0, holomorphic=True))

print("NetKet 的 QGT:")
print(f"  类型: {type(qgt_netket)}")

# 获取 QGT 矩阵
qgt_netket_dense = qgt_netket.to_dense()
print(f"  形状: {qgt_netket_dense.shape}")
print(f"  trace: {jnp.trace(qgt_netket_dense):.6e}")
print(f"  条件数: {jnp.linalg.cond(qgt_netket_dense):.6e}")
print(f"  对角线元素范围: [{jnp.min(jnp.abs(jnp.diag(qgt_netket_dense))):.6e}, {jnp.max(jnp.abs(jnp.diag(qgt_netket_dense))):.6e}]")

## 根本原因分析

In [ ]:
print("\n" + "="*70)
print("根本原因分析")
print("="*70)
print()

print("如果 QGT trace = 0，可能的原因：")
print()
print("1. 梯度计算问题：")
print("   - holomorphic=True 可能不适用于当前网络")
print("   - 需要检查复数梯度的正确性")
print()
print("2. 波函数退化：")
print("   - 网络可能输出常数或接近常数的值")
print("   - 所有样本的 logψ 几乎相同")
print()
print("3. 参数初始化问题：")
print("   - 初始化可能导致网络陷入退化状态")
print("   - 需要检查网络输出")
print()

# 检查网络输出
log_psi_samples = forward(GraphDef_State, samples)
print("网络输出检查：")
print(f"  logψ 范围: [{jnp.min(log_psi_samples.real):.6f}, {jnp.max(log_psi_samples.real):.6f}]")
print(f"  logψ 虚部范围: [{jnp.min(log_psi_samples.imag):.6f}, {jnp.max(log_psi_samples.imag):.6f}]")
print(f"  logψ 标准差: {jnp.std(log_psi_samples):.6e}")

if jnp.std(log_psi_samples) < 1e-6:
    print("\n⚠️ 严重问题：网络输出几乎为常数！")
    print("   这就是 QGT trace = 0 的根本原因！")
    print("   解决方案：重新初始化网络或改变网络架构")

print("\n" + "="*70)

## 修复方案：正确的 QGT 计算

In [ ]:
from jax import flatten_util

@partial(jax.jit, static_argnames=("model_forward", "graphdef"))
def compute_qgt_correct(model_forward, graphdef, state, samples):
    """
    正确的 QGT 计算
    
    关键修复：
    1. 使用 flatten_util.ravel_pytree 正确展平
    2. 正确处理复数梯度
    3. 确保数值稳定性
    """
    n_samples = samples.shape[0]
    
    # 定义 logpsi 函数
    def logpsi_single_param(s, x):
        return model_forward((graphdef, s), x)
    
    # 对每个样本计算梯度
    def grad_logpsi_single(x):
        grad = jax.grad(logpsi_single_param, argnums=0, holomorphic=True)(state, x)
        # 使用 flatten_util.ravel_pytree 展平
        flat_grad, _ = flatten_util.ravel_pytree(grad)
        return flat_grad
    
    # 批量计算
    grads_flat = jax.vmap(grad_logpsi_single)(samples)
    
    print(f"展平梯度形状: {grads_flat.shape}")
    print(f"展平梯度范数: {jnp.linalg.norm(grads_flat):.6e}")
    
    # 计算均值和中心化
    mean_grad = jnp.mean(grads_flat, axis=0)
    centered_grads = grads_flat - mean_grad
    
    print(f"中心化梯度范数: {jnp.linalg.norm(centered_grads):.6e}")
    
    # 计算 QGT
    qgt = (1.0 / n_samples) * jnp.einsum('si,sj->ij', 
                                          jnp.conj(centered_grads), 
                                          centered_grads)
    
    return qgt

# 测试修复后的 QGT
qgt_fixed = compute_qgt_correct(forward, graphdef, model_state, samples)

print(f"\n修复后的 QGT:")
print(f"  trace: {jnp.trace(qgt_fixed):.6e}")
print(f"  条件数: {jnp.linalg.cond(qgt_fixed):.6e}")